# 🌊 AegisFlow AI — Flood Risk Model Training
**RandomForest Classifier** dự báo nguy cơ ngập lụt tại Đà Nẵng

- Dataset: 8,000 mẫu thực tế 2019–2024
- Features: water_level, rainfall, tide, historical_score, hours_rain
- Output: 4 mức rủi ro (low / medium / high / critical)

## 📦 1. Cài đặt thư viện

In [ ]:
!pip install scikit-learn joblib numpy pandas matplotlib seaborn -q
print('✅ Libraries installed')

## 📂 2. Load & Khám phá Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Load dataset
URL = 'https://raw.githubusercontent.com/IzooBeeee/aegisflow-ai-service-/main/data/flood_danang_clean_balanced.csv'
df = pd.read_csv(URL)

print('=' * 55)
print('  AegisFlow AI — Flood Dataset (Đà Nẵng 2019–2024)')
print('=' * 55)
print(f'  Tổng số mẫu   : {len(df):,}')
print(f'  Số features   : {df.shape[1]}')
print(f'  Phân bố nhãn  : {dict(Counter(df["risk_level"]))}')
print()
df.head(10)

In [ ]:
# Thống kê mô tả
FEATURES = ['water_level_m', 'rainfall_mm', 'hours_rain', 'tide_level', 'historical_score']
print('📊 Thống kê các features:')
df[FEATURES].describe().round(3)

In [ ]:
# Visualize phân bố nhãn
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Pie chart
counts = df['risk_level'].value_counts()
colors = ['#22d3ee', '#f59e0b', '#f97316', '#ef4444']
axes[0].pie(counts.values, labels=counts.index, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Phân bố mức rủi ro (8,000 mẫu)', fontweight='bold')

# Feature boxplot
df_melt = df[['water_level_m', 'rainfall_mm', 'risk_level']].melt(id_vars='risk_level')
sns.boxplot(data=df[FEATURES[:3]], ax=axes[1], palette='Set2')
axes[1].set_title('Phân bố features chính', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
print('✅ Dataset trực quan hóa xong')

## 🤖 3. Huấn luyện RandomForest Model

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from datetime import datetime

# Chuẩn bị data
X = df[FEATURES].values
y = df['risk_level'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Train: {len(X_train):,} mẫu | Test: {len(X_test):,} mẫu')
print(f'Classes: {list(le.classes_)}')
print()

# Class weights
label_counts = Counter(y_train)
n_samples = len(y_train)
n_classes = len(label_counts)
class_weight = {cls: n_samples / (n_classes * cnt) for cls, cnt in label_counts.items()}

# Train model
print('🚀 Bắt đầu huấn luyện RandomForest (200 cây)...')
t0 = datetime.now()

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight=class_weight,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
model.fit(X_train, y_train)

elapsed = (datetime.now() - t0).total_seconds()
print(f'\n✅ Training hoàn tất trong {elapsed:.1f}s')

## 📈 4. Đánh giá Model

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print('=' * 55)
print('  KẾT QUẢ ĐÁNH GIÁ MODEL')
print('=' * 55)
print(f'  Accuracy : {acc:.4f} ({acc*100:.2f}%)')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Cross-validation
print('🔄 Chạy 5-fold Cross-Validation...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y_enc, cv=cv, scoring='f1_weighted')
print(f'CV F1 (weighted): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Scores: {[round(s,4) for s in cv_scores]}')

In [ ]:
# Visualize kết quả
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f'Confusion Matrix\nAccuracy: {acc*100:.2f}%', fontweight='bold')
axes[0].set_ylabel('Thực tế')
axes[0].set_xlabel('Dự đoán')

# Feature importance
importances = model.feature_importances_
feat_labels = ['Water Level', 'Rainfall', 'Hours Rain', 'Tide Level', 'Historical Score']
sorted_idx = np.argsort(importances)[::-1]
colors_bar = ['#22d3ee', '#a78bfa', '#34d399', '#f59e0b', '#f87171']
axes[1].bar(range(len(FEATURES)), importances[sorted_idx],
            color=[colors_bar[i] for i in sorted_idx])
axes[1].set_xticks(range(len(FEATURES)))
axes[1].set_xticklabels([feat_labels[i] for i in sorted_idx], rotation=15)
axes[1].set_title('Feature Importance', fontweight='bold')
axes[1].set_ylabel('Importance Score')

plt.tight_layout()
plt.show()

## 💾 5. Lưu Model

In [ ]:
import joblib, json

joblib.dump({
    'model': model,
    'label_encoder': le,
    'feature_names': FEATURES,
    'version': 'v4.1',
    'accuracy': float(acc),
    'trained_at': datetime.now().isoformat(),
}, 'flood_risk_model.pkl')

print('=' * 55)
print('  ✅ TRAINING HOÀN TẤT')
print('=' * 55)
print(f'  Model    : RandomForest v4.1 (200 cây)')
print(f'  Dataset  : 8,000 mẫu — Đà Nẵng 2019–2024')
print(f'  Accuracy : {acc*100:.2f}%')
print(f'  Classes  : {list(le.classes_)}')
print(f'  File     : flood_risk_model.pkl')
print()
print('🚀 Model sẵn sàng deploy lên AegisFlow AI Service!')